<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Analyze%20customer%20transaction%20data%20relevant%20to%20Kenyan%20microfinance%20institutions%20to%20group%20clients%20by%20behavior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Create Customers data (using local filesystem as HDFS is not available)
!mkdir -p /mfi && mv customers.csv /mfi/
!ls /mfi && cat /mfi/customers.csv | head -5

customers.csv
cust_id,loans,avg_loan_ksh,on_time_pct,savings_ksh,txns_month,dpd
1,5,50000,98,120000,12,0
2,8,35000,95,80000,10,2
3,2,10000,60,5000,2,45
4,6,45000,92,95000,11,3


In [4]:
%%writefile customers.csv
cust_id,loans,avg_loan_ksh,on_time_pct,savings_ksh,txns_month,dpd
1,5,50000,98,120000,12,0
2,8,35000,95,80000,10,2
3,2,10000,60,5000,2,45
4,6,45000,92,95000,11,3
5,1,15000,70,8000,3,30
6,9,60000,97,150000,14,0
7,3,20000,55,3000,2,60
8,7,40000,88,70000,9,5
9,4,25000,75,20000,6,15
10,10,55000,96,140000,13,1
11,2,12000,65,6000,2,40
12,5,30000,85,50000,8,7
13,1,8000,50,2000,1,75
14,6,38000,90,75000,10,4
15,3,18000,72,15000,5,20

Writing customers.csv


In [8]:
# Spark clustering: with vs without feature selection

# Install PySpark
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import pyspark.sql.functions as F

# Initialize SparkSession
spark = SparkSession.builder.appName("ClusteringExample").getOrCreate()

# Read data from local filesystem (HDFS not available)
df = spark.read.option("header", True).option("inferSchema", True) \
        .csv("/mfi/customers.csv")      # <-- reads from local filesystem
df = df.withColumn("noise", rand(42))                        # deliberately useless feature

GOOD = ["on_time_pct","savings_ksh","txns_month","dpd","avg_loan_ksh"]      # curated
BAD  = ["cust_id","loans","avg_loan_ksh","on_time_pct","savings_ksh",
        "txns_month","dpd","noise"]                                          # everything + noise

def run(input_df, cols, k=3):
    current_df = input_df
    # Ensure all columns are numeric for VectorAssembler
    for c in cols:
        if current_df.schema[c].dataType.simpleString() not in ('double', 'float', 'integer', 'long'):
            current_df = current_df.withColumn(c, col(c).cast("double"))

    v = VectorAssembler(inputCols=cols, outputCol="f").transform(current_df)
    s = StandardScaler(inputCol="f", outputCol="s").fit(v).transform(v)
    r = KMeans(k=k, seed=42, featuresCol="s").fit(s).transform(s)
    return r, ClusteringEvaluator(featuresCol="s", metricName="silhouette").evaluate(r)

r_bad,  s_bad  = run(df, BAD)
r_good, s_good = run(df, GOOD)
print("silhouette with noisy/all features:", round(s_bad,3))   # lower, muddled groups
print("silhouette with selected features :", round(s_good,3))  # higher, clean groups

r_good.groupBy("prediction").agg(F.count("*").alias("n"),
    *[F.round(F.mean(c),1).alias(c) for c in GOOD],
    F.collect_list("cust_id").alias("clients")).orderBy("prediction").show(truncate=False)

silhouette with noisy/all features: 0.561
silhouette with selected features : 0.604
+----------+---+-----------+-----------+----------+----+------------+---------------------+
|prediction|n  |on_time_pct|savings_ksh|txns_month|dpd |avg_loan_ksh|clients              |
+----------+---+-----------+-----------+----------+----+------------+---------------------+
|0         |4  |95.8       |126250.0   |12.5      |1.0 |52500.0     |[1, 4, 6, 10]        |
|1         |5  |60.0       |4800.0     |2.0       |50.0|13000.0     |[3, 5, 7, 11, 13]    |
|2         |6  |84.2       |51666.7    |8.0       |8.8 |31000.0     |[2, 8, 9, 12, 14, 15]|
+----------+---+-----------+-----------+----------+----+------------+---------------------+

